# HPC4e heterogeneous mixed elasticity

Devloo et al. (2021), section 6.2, DOI [10.1051/m2an/2021013](https://doi.org/10.1051/m2an/2021013).
The original geological material samples, 16 × 8 macrorectangles, RT1/Q1/P1 local spaces and 32 × 32 local rectangles match the published experiment. Only the number of P1 subfaces changes: 1, 2, 4, 8. The expected stress-profile oscillations decrease under this skeleton refinement.

This notebook reads the archived research acquisitions; the final cell solves a small affine mixed-elasticity patch. Full geological solves and classical RT reference refinement are separate from lightweight automated tests.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

root = Path.cwd()
if not (root / "examples").exists():
    root = root.parent
records = root / "examples/results/hpc4e"
manifest = json.loads((records / "dataset.json").read_text())
print(manifest["repository"], manifest["revision"])
print("Material grid:", manifest["shape"], "; bounds [m]:", manifest["physical_bounds_m"])
for segments in (1, 2, 4, 8):
    row = json.loads((records / f"mhm-s{segments}.json").read_text())
    print(segments, "subfaces:", row["unknown_global_dofs"], "global unknowns", row["diagnostics"])

## Published profile and independent reference

The profile is at z = 2250.25 m. Digitization intervals contain the visible line thickness plus two pixels vertically and horizontally. Their overlap checks agreement with the image, not an integrated error bound. Independently assembled DOLFINx/UFL RT1 and RT2 fields on 512 × 256 cells check polynomial-degree sensitivity; RT2 on 1024 × 512 checks spatial refinement. The fine RT2 acquisition uses four MPI ranks and pivoted MUMPS LDLᵀ with out-of-core factors. Weak stress symmetry is a moment constraint; all stress norms use the full tensor.


In [ ]:
import math
comparison = json.loads((records / "complementary-energy.json").read_text())
for row in comparison["rows"]:
    energy = row["energy"]
    print("subfaces", row["segments"], "stress L2 [%]", 100 * row["l2"]["stress_relative"],
          "displacement L2 [%]", 100 * row["l2"]["displacement_relative"],
          "compliance [%]", 100 * math.sqrt(
              energy["stress_compliance_difference_squared"] / energy["reference_energy"]),
          "energy identity relative defect", energy["identity_relative_difference"])
comparisons = json.loads((records / "published-profile-comparison.json").read_text())
for row in comparisons:
    print("subfaces", row["segments"], "overlap", row["interval_overlap_count"], "/", row["samples"],
          "nominal RMS [MPa]", row["nominal_rms_difference_mpa"])
for filename in ("material.png", "published-profile-comparison.png", "stress-profiles.png",
                 "stress-displacement-fields.png", "displacement-vectors.png"):
    display(Image(filename=str(root / "docs/figures/hpc4e" / filename)))

## Classical reference sensitivity

The RT2 spatial increment from 512 × 256 to 1024 × 512 is 1.119129% in stress L2, 0.010589% in displacement L2, 2.864135% in rotation L2 and 3.113395% in compliance norm. These increments use the fine reference norm as denominator and are not continuum error bounds. The finest MHM field differs from that reference by 4.159399%, 0.625204%, 8.202742% and 7.338907%, respectively. Reference sensitivity remains relevant, particularly for rotation and compliance.

The separate RT1-to-RT2 degree increment on 512 × 256 is retained in its own numerical record. Orders four and five check all field-norm integrals on the common, material-aligned partition. The figure manifest ties the displayed images to the reference and MHM input digests.


In [ ]:
import hashlib

refinement = json.loads((records / "reference-spatial-refinement.json").read_text())
assert refinement["source_changed"] is False
pairs = {}
for row in refinement["rows"]:
    pairs.setdefault(row["approximation"], {})[row["norms"]["quadrature_order"]] = row["norms"]
for name, pair in pairs.items():
    for field in ("stress", "displacement", "rotation", "compliance"):
        key = field + "_difference_dimensionless"
        assert abs(pair[4][key] - pair[5][key]) <= 1e-10 * pair[5][key]
    print(name, {field: 100 * pair[5][field + "_relative"]
          for field in ("stress", "displacement", "rotation", "compliance")})
for name in ("classical-rt1-512x256-pardiso", "classical-rt2-512x256-mumps",
             "classical-rt2-1024x512-mumps"):
    reference = json.loads((records / (name + ".json")).read_text())
    print(name, "DOFs", reference["native_dofs"], "original residual", reference["residual"],
          "force equilibrium", reference["equilibrium_relative"],
          "energy/work", reference["energy_work_relative"])
visualization = json.loads((records / "visualization.json").read_text())
assert visualization["reference"] == "classical-rt2-1024x512-mumps.npz"
assert visualization["input_sha256"][visualization["reference"]] == reference["archive_sha256"]
for name, expected in visualization["figure_sha256"].items():
    assert hashlib.sha256((root / "docs/figures/hpc4e" / name).read_bytes()).hexdigest() == expected
print("Six figure digests verified; field samples preserve owning-cell values.")
display(Image(filename=str(root / "docs/figures/hpc4e/reference-sensitivity.png")))


## Small exact patch

The affine displacement u = (x, −z) is represented exactly. Its trace-free strain makes the field compatible with both finite and incompressible bulk response; this patch checks the operator and the three rigid modes, not general locking-free convergence.

In [ ]:
import numpy as np
from pymhm import CartesianMacroMesh, solve_elasticity_tensor_rt

mesh = CartesianMacroMesh(2, 1, (0, 1, 0, 0.45))
exact = lambda x: np.column_stack((x[:, 0], -x[:, 1]))
solution = solve_elasticity_tensor_rt(mesh, local_refinement=2, dirichlet=exact, lame_lambda=1e8)
assert np.max(np.abs(solution.equilibrium_residuals())) < 1e-9
assert np.max(np.abs(solution.fine_force_residuals())) < 1e-9
print("Force and rigid-moment equilibrium:", np.max(np.abs(solution.equilibrium_residuals())))